# Albanian Style Model - Gemma 4 Training Pipeline
This notebook provides an end-to-end pipeline for training the **Gemma 4:e4b** model on the stylistic elements of Albanian text registers. It uses the seed data from the Lahuta project to fine-tune the model for high-fidelity style imitation.

**Hardware Requirement:** GPU (T4, L4, A100, or V100 recommended).

## 1. Setup Environment
We use **Unsloth** for 2x faster and 70% less memory usage during fine-tuning.

In [ ]:
%%capture
!pip install --no-deps "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps "xformers<0.0.27" "trl<0.9.0" peft accelerate bitsandbytes
!pip install datasets

## 2. Load Model
We will load the **Gemma 4:e4b** base model (using the identifier `google/gemma-4-1b-it` for the base weights).

In [ ]:
from unsloth import FastLanguageModel
import torch
max_seq_length = 2048 
dtype = None 
load_in_4bit = True 

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "google/gemma-4-1b-it",
    max_seq_length = max_seq_length,
    dtype = dtype,
    load_in_4bit = load_in_4bit,
)

## 3. Prepare Data
We will load the seed data and format it using the **Gemma 4 / Lahuta** prompt format:
```
<start_of_turn>system
{system_message}<end_of_turn>
<start_of_turn>user
{user_message}<end_of_turn>
<start_of_turn>model
{model_response}<end_of_turn>
```

In [ ]:
import os
from datasets import Dataset

def load_seed_data(base_path="data/seed"):
    texts = []
    # Walk through registers (editorial, informational, etc.)
    for register in os.listdir(base_path):
        register_path = os.path.join(base_path, register)
        if os.path.isdir(register_path):
            for filename in os.listdir(register_path):
                if filename.endswith(".txt"):
                    with open(os.path.join(register_path, filename), "r", encoding="utf-8") as f:
                        content = f.read()
                        texts.append({
                            "register": register,
                            "content": content
                        })
    return Dataset.from_list(texts)

dataset = load_seed_data()

GEMMA_PROMPT = """<start_of_turn>system
Je një redaktor ekspert i gjuhës shqipe që specializohet në stilin {}.<end_of_turn>
<start_of_turn>user
Shkruaj një artikull shembull në stilin {}.<end_of_turn>
<start_of_turn>model
{}<end_of_turn>"""

def formatting_prompts_func(examples):
    registers = examples["register"]
    contents  = examples["content"]
    texts = []
    for register, content in zip(registers, contents):
        text = GEMMA_PROMPT.format(register, register, content)
        texts.append(text)
    return { "text" : texts, }

dataset = dataset.map(formatting_prompts_func, batched = True,)

## 4. Setup LoRA Training
Gemma 4 attention projections are targeted for fine-tuning.

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = 16, 
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj",
                      "gate_proj", "up_proj", "down_proj",],
    lora_alpha = 16,
    lora_dropout = 0, 
    bias = "none",    
    use_gradient_checkpointing = "unsloth", 
    random_state = 3407,
    use_rslora = False,  
    loftq_config = None, 
)

## 5. Execute Training

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    dataset_text_field = "text",
    max_seq_length = max_seq_length,
    dataset_num_proc = 2,
    packing = False, 
    args = TrainingArguments(
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        max_steps = 100,
        learning_rate = 2e-4,
        fp16 = not torch.cuda.is_bf16_supported(),
        bf16 = torch.cuda.is_bf16_supported(),
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        output_dir = "outputs",
    ),
)

trainer_stats = trainer.train()

## 6. Save Model

In [ ]:
model.save_pretrained("albanian_style_gemma_lora")
tokenizer.save_pretrained("albanian_style_gemma_lora")

## 7. Inference
Test the trained model's ability to imitate the target styles.

In [ ]:
FastLanguageModel.for_inference(model) 

register = "editorial"
inputs = tokenizer(
[
    GEMMA_PROMPT.format(
        register, 
        register, 
        "", 
    ).replace("<end_of_turn>", "") # Start generation at model turn
], return_tensors = "pt").to("cuda")

outputs = model.generate(**inputs, max_new_tokens = 512, use_cache = True)
print(tokenizer.batch_decode(outputs))